# Fetch Laser Execution Data

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.storage.health_pb2 import HealthDto, HealthRequest
from femtum_sdk.api import DatastoreApi

api_host = "localhost:5208"

with DatastoreApi(host=api_host) as api:
    result: HealthDto = api.health.GetHealth(HealthRequest())
    print(result)


## Fetch Laser Executions Data

### By Page

In [ ]:
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import LaserExecutionPage, ListByPageLaserExecutionsRequest

with DatastoreApi(host=api_host) as api:
    page: LaserExecutionPage = api.laser_execution.ListByPage(ListByPageLaserExecutionsRequest(
        page_size=10,
        page_number=0,
    ))

    print(page)


### By Page with filter (For example component path)

In [ ]:
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import LaserExecutionsFilterRequest
from femtum_sdk.api.contracts.v1.storage.query_pb2 import QueryField, QueryOperator


with DatastoreApi(host=api_host) as api:
    page: LaserExecutionPage = api.laser_execution.ListByPage(ListByPageLaserExecutionsRequest(filters=LaserExecutionsFilterRequest(component_path=QueryField(value="Bob", operator=QueryOperator.EQUALS))))

    print(page)

## Fetch Images Attached to Execution

Fetch image data

In [ ]:
from femtum_sdk.api.contracts.v1.dto.attachment_pb2 import Attachment
from femtum_sdk.api.contracts.v1.dto.laser_execution_pb2 import LaserExecution
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import FindLaserExecutionByIdRequest

with DatastoreApi(host=api_host) as api:
    laser_execution: LaserExecution = api.laser_execution.FindById(FindLaserExecutionByIdRequest(id="155"))

    print(f"Recipe ID: {laser_execution.recipe_id}")
    print(f"Component ID: {laser_execution.component_id}")
    print(f"Calibration ID: {laser_execution.laser_energy_calibration_id}")

    print(laser_execution.attachments)
    attachment_ids = [attachment.id for attachment in laser_execution.attachments]

    attachments: list[Attachment] = []
    for attachment_id in attachment_ids:
        attachment = api.attachment.Download(id=attachment_id)
        attachments.append(attachment)

Display images

In [ ]:
from IPython.display import display, Image, Markdown

for attachment in attachments:
    if attachment.mime_type.startswith("image/"):
        display(Markdown(f"**{attachment.name}**"))
        display(Image(data=attachment.blob_data, format=attachment.mime_type.split("/")[-1]))
    else:
        print(f"Skipping non-image attachment: {attachment.name} ({attachment.mime_type})")